In [1]:
import sqlite3
import pandas as pd 
import numpy as np
from datetime import datetime, timedelta


In [2]:
conn=sqlite3.connect('unisole_analytics.db')
cursor=conn.cursor()

In [5]:
cursor.execute('''
CREATE TABLE IF NOT EXISTS 
historical_rides(
               ride_id INTEGER,
               user_id TEXT,
               distance REAL,
               price REAL,
               ride_time_minutes INTEGER,
               rating INTEGER,
               date DATE
)
''')

data = []
start_date=datetime(2025,1,1)
for i in range(10000):
    data.append((i,f"User_{np.random.randint(1,500)}",
                 round(np.random.uniform(2,50),2),
                 round(np.random.uniform(100,1000),2),
                 np.random.randint(10,120),
                 np.random.randint(1,6),
                 (start_date +
                  timedelta(days=np.random.randint(0,365))).strftime('%Y-%m-%d')
             ))
cursor.executemany(
        "INSERT INTO historical_rides VALUES (?,?,?,?,?,?,?)",
        data
        )
conn.commit()
print("OLAP table populated with 10,000 historical records")
    


OLAP table populated with 10,000 historical records


In [7]:
#monthnly revenue analysis
query= """
SELECT
  strftime("%m",date) AS Month, 
  COUNT(ride_id) AS Total_Rides,
  SUM(price) AS Total_revenue,
  AVG(rating) AS Avg_Rating
  FROM historical_rides
GROUP BY  Month
ORDER BY Month
"""
monthly_report= pd.read_sql_query(query,conn)
print("monthly business insights")
print("monthly_report")

monthly business insights
monthly_report


In [8]:
# ML Feature Correlation: Distance vs Price

df_ml = pd.read_sql_query(
    "SELECT distance, price FROM historical_rides",
    conn
)

correlation = df_ml['distance'].corr(df_ml['price'])

print(f"Correlation between Distance and Price: {correlation:.2f}")

# Industry Note:
# Agar ye millions of rows hote,
# toh hum Parquet file use karte (Column-major)



Correlation between Distance and Price: -0.01


In [9]:
query = """
SELECT
    user_id,
    SUM(price) AS lifetime_value,
    RANK() OVER (
        ORDER BY SUM(price) DESC
    ) AS user_rank
FROM historical_rides
GROUP BY user_id
LIMIT 10;
"""

rankings = pd.read_sql_query(query, conn)

print("--- Top 10 High-Value Users (LTV) ---")
print(rankings)


--- Top 10 High-Value Users (LTV) ---
    user_id  lifetime_value  user_rank
0  User_212        35379.61          1
1  User_339        33128.00          2
2  User_463        32402.85          3
3  User_471        32088.07          4
4  User_405        31758.78          5
5   User_17        31692.24          6
6  User_217        31328.66          7
7  User_441        30841.53          8
8  User_278        30711.37          9
9  User_456        30657.10         10


In [10]:
# Creating a training set with specific filters

training_query = """
SELECT
    distance,
    ride_time_minutes,
    price
FROM historical_rides
WHERE rating >= 4
AND price > 200;
"""

train_df = pd.read_sql_query(training_query, conn)

print(f"Training set ready with {len(train_df)} samples.")
print(train_df.head())



Training set ready with 7117 samples.
   distance  ride_time_minutes   price
0     21.24                119  534.32
1     29.68                100  874.03
2      4.73                 95  741.95
3     21.18                110  653.34
4     39.69                 84  385.29
